# Topic 03 — Quantization trade-offs

One immutable Qwen2.5-0.5B-Instruct snapshot; two separate loads (FP16 and bitsandbytes NF4); paired prompt/output contract. This is a **bounded smoke experiment**, not a production quality benchmark. The notebook does not publish measured results. Run it on a CUDA GPU and preserve the JSON evidence.

Set `LSM_MODEL_CACHE` to a persistent local directory (for example a mounted drive) before the first download if the runtime is ephemeral. The model is fetched once; subsequent cells read the same local snapshot with no network. Never commit this cache. A free Colab GPU or a particular GPU type is not guaranteed.

In [ ]:
%pip install -q 'transformers==4.51.3' 'accelerate==1.6.0' 'bitsandbytes==0.50.2' 'huggingface_hub==0.30.2' 'safetensors==0.5.3'


In [ ]:
import gc, json, math, os, platform, statistics, time
from pathlib import Path
import torch, transformers, bitsandbytes as bnb
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.generation.streamers import BaseStreamer
assert torch.cuda.is_available(), 'Measured comparison requires a CUDA GPU; do not fabricate results'
assert torch.cuda.get_device_capability()[0] >= 6, 'NF4 path requires CUDA compute capability 6.0+'
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
CACHE = Path(os.environ.get('LSM_MODEL_CACHE', str(Path.cwd() / '.cache' / 'models'))).expanduser().resolve()
CACHE.mkdir(parents=True, exist_ok=True)
try:
    MODEL_DIR = Path(snapshot_download(repo_id=MODEL_ID, revision=REVISION, cache_dir=str(CACHE), local_files_only=True))
    print('Reusing complete pinned snapshot')
except FileNotFoundError:
    print('One-time download of pinned snapshot')
    MODEL_DIR = Path(snapshot_download(repo_id=MODEL_ID, revision=REVISION, cache_dir=str(CACHE)))
assert (MODEL_DIR / 'config.json').exists() and list(MODEL_DIR.glob('*.safetensors')) and (MODEL_DIR / 'tokenizer.json').exists()
print('Using pinned local snapshot:', MODEL_DIR, 'GPU:', torch.cuda.get_device_name(0))


## Before measuring: a hand prediction

For a toy 3.0B dense model, FP16 weight payload is 6.0 decimal GB. A perfect 4-bit payload is 1.5 GB; 128-element groups with one FP16 scale add 46.875 MB, before zeros, unquantized modules, padding and runtime memory. Write your prediction for the actual 0.5B model and state what this model omits. NF4 uses a non-uniform codebook, so the scalar symmetric INT4 arithmetic in the lecture is *not* its implementation.

In [ ]:
params = 3_000_000_000
toy = {'fp16_gb': params * 2 / 1e9, 'int4_payload_gb': params / 2 / 1e9, 'fp16_scales_mb': math.ceil(params / 128) * 2 / 1e6}
assert toy == {'fp16_gb': 6.0, 'int4_payload_gb': 1.5, 'fp16_scales_mb': 46.875}
toy


## Paired workloads and timing contract

`TokenClock` records the first generated token callback after synchronizing CUDA. The returned total wall time includes Python/host overhead; `(total−TTFT)/(output_tokens−1)` is an approximate observed post-first interval, not GPU-only kernel time. Every format uses the same prompts, tokenizer, deterministic decoding and three warmed trials. A tiny sample has unstable p95; report raw trials and never generalize the smoke quality set.

In [ ]:
PROMPTS = [
    {'bucket':'format', 'text':'Reply with exactly one word: blue or orange. What color is the sky on a clear day?'},
    {'bucket':'arithmetic', 'text':'Compute 17 + 28. Give only the integer.'},
    {'bucket':'instruction', 'text':'Write exactly three comma-separated verbs for measuring a service.'},
    {'bucket':'reasoning', 'text':'A server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.'},
]
TRIALS, MAX_NEW = 3, 24
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
class TokenClock(BaseStreamer):
    def __init__(self): self.seen_prompt = False; self.first = None
    def put(self, value):
        if not self.seen_prompt: self.seen_prompt = True; return
        torch.cuda.synchronize()
        if self.first is None: self.first = time.perf_counter()
    def end(self): pass
def p95(values): return sorted(values)[math.ceil(.95 * len(values)) - 1]
def run_one(model, prompt):
    ids = tokenizer.apply_chat_template([{'role':'user','content':prompt}], tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    clock = TokenClock(); torch.cuda.synchronize(); start = time.perf_counter()
    with torch.inference_mode():
        result = model.generate(ids, streamer=clock, do_sample=False, max_new_tokens=MAX_NEW, min_new_tokens=MAX_NEW, pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize(); end = time.perf_counter()
    output_tokens = result.shape[-1] - ids.shape[-1]
    assert clock.first is not None and output_tokens > 0
    return {'prompt_tokens':int(ids.shape[-1]), 'output_tokens':int(output_tokens), 'ttft_s':clock.first-start, 'wall_s':end-start, 'post_first_s_per_token':(end-clock.first)/max(1, output_tokens-1), 'output':tokenizer.decode(result[0,ids.shape[-1]:], skip_special_tokens=True)}


In [ ]:
RUN = {'contract':'topic-03-paired-smoke-v1', 'utc_started':time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'model_id':MODEL_ID, 'revision':REVISION, 'snapshot':str(MODEL_DIR), 'gpu':torch.cuda.get_device_name(0), 'compute_capability':torch.cuda.get_device_capability(0), 'torch':torch.__version__, 'cuda':torch.version.cuda, 'transformers':transformers.__version__, 'bitsandbytes':bnb.__version__, 'trials':TRIALS, 'max_new_tokens':MAX_NEW, 'formats':{}}
for label in ('fp16', 'nf4_w4a16'):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16) if label == 'nf4_w4a16' else None
    load_start = time.perf_counter()
    model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), local_files_only=True, torch_dtype=torch.float16, quantization_config=quant, device_map={'':0}).eval()
    torch.cuda.synchronize(); load_s = time.perf_counter() - load_start
    _ = run_one(model, PROMPTS[0]['text'])  # warm-up outside trial population
    rows = []
    for case in PROMPTS:
        trials = [run_one(model, case['text']) for _ in range(TRIALS)]
        rows.append({'bucket':case['bucket'], 'prompt':case['text'], 'trials':trials, 'p95_ttft_s':p95([r['ttft_s'] for r in trials]), 'p95_wall_s':p95([r['wall_s'] for r in trials])})
    RUN['formats'][label] = {'load_s':load_s, 'peak_allocated_bytes':torch.cuda.max_memory_allocated(), 'peak_reserved_bytes':torch.cuda.max_memory_reserved(), 'rows':rows}
    del model; gc.collect(); torch.cuda.empty_cache()
    print(label, 'complete; max reserved GiB:', round(RUN['formats'][label]['peak_reserved_bytes'] / 2**30, 3))
assert set(RUN['formats']) == {'fp16','nf4_w4a16'}


## Review and export

Compare the outputs prompt by prompt. Record **valid**, **incorrect**, **truncated** or **needs human review** for each format in a separate analysis file; do not score semantic quality by blindly comparing strings. For a real rollout, repeat on a representative held-out task set, include uncertainty and per-bucket failures, verify the actual kernel path, and compare under the intended concurrency. `NF4` here is a bitsandbytes weight-only path, not AWQ/GPTQ and not proof of speedup in vLLM.

In [ ]:
for i, case in enumerate(PROMPTS):
    print('\nBUCKET:', case['bucket'], '| PROMPT:', case['text'])
    for label in ('fp16','nf4_w4a16'):
        print(label, '=>', RUN['formats'][label]['rows'][i]['trials'][0]['output'])
evidence = Path('evidence'); evidence.mkdir(exist_ok=True)
target = evidence / 'topic-03-comparison.json'
target.write_text(json.dumps(RUN, indent=2, default=str), encoding='utf-8')
print('Saved:', target)


## Decision memo

In `evidence/topic-03-decision.md`: (1) hand-calculate expected payload and explain scale/metadata omissions; (2) compare measured *process* memory, cold load, raw TTFT/wall/post-first intervals and outputs; (3) identify at least one surprising regression or failed assumption; (4) decide whether NF4 is acceptable for this hardware and workload and name the next experiment. Do **not** transfer these numbers to the instructor RTX 5070 Ti without a new run. Sources: [AWQ](https://arxiv.org/abs/2306.00978), [GPTQ](https://arxiv.org/abs/2210.17323), [bitsandbytes](https://huggingface.co/docs/transformers/main/quantization/bitsandbytes), [vLLM supported formats](https://docs.vllm.ai/en/stable/features/quantization/).